# OpenEnv Agentic Issue Workflow (Colab)

> Purpose: run issue work in a repeatable Red -> Green -> Review -> PR loop with artifacts.

Workflow modules:
1. Configure target issue and branch
2. Bootstrap environment
3. Ingest issue context and produce a brief
4. Run tests and implement changes
5. Run validation gates
6. Generate PR draft artifacts

In [ ]:
# Module 1: configuration
REPO_URL = "https://github.com/soham2710/OpenEnv.git"
BRANCH = "test/hf-deploy-script-coverage"
WORKDIR = "OpenEnv"

# Issue metadata (edit these per issue)
ISSUE_NUMBER = 54
ISSUE_TITLE = "Break up deployment script in #49"
ISSUE_URL = f"https://github.com/meta-pytorch/OpenEnv/issues/{ISSUE_NUMBER}"

# Test/validation targets
PYTEST_TARGET = "tests/scripts/test_prepare_hf_deployment.py"
PYTEST_FLAGS = "-q"
RUN_LINT = False
RUN_FULL_TESTS = False

In [ ]:
# Module 2: shared helpers
import json
import os
import re
import subprocess
from pathlib import Path
from urllib.request import urlopen

def run(cmd: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=check, text=True, capture_output=False)

def run_capture(cmd: str) -> str:
    print(f"$ {cmd}")
    out = subprocess.check_output(cmd, shell=True, text=True)
    print(out)
    return out

def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    print(f"Wrote {path}")

def fetch_issue_text(url: str) -> str:
    try:
        html = urlopen(url).read().decode("utf-8", errors="ignore")
    except Exception as exc:
        return f"Failed to fetch issue page: {exc}"
    title_match = re.search(r"<title>(.*?)</title>", html, flags=re.IGNORECASE | re.DOTALL)
    title = title_match.group(1).strip() if title_match else "Unknown title"
    return f"Issue URL: {url}\nPage title: {title}\n"

In [ ]:
# Module 3: bootstrap repo and toolchain
workspace = Path('/content') / WORKDIR
if workspace.exists():
    run(f"rm -rf {workspace}")

run(f"git clone -b {BRANCH} {REPO_URL} {WORKDIR}")
os.chdir(workspace)
run("python -m pip install -q uv")
run("uv sync --group dev")

print("Setup complete")
run_capture("git status --short --branch")
run_capture(".venv/bin/python --version")
run_capture(".venv/bin/uv --version")

In [ ]:
# Module 4: issue intake and brief artifact
os.chdir(f"/content/{WORKDIR}")
issue_text = fetch_issue_text(ISSUE_URL)

brief = f"""# Issue Brief
Issue: #{ISSUE_NUMBER}
Title: {ISSUE_TITLE}
URL: {ISSUE_URL}

## Context snapshot
{issue_text}

## Proposed plan
1. Write/extend failing tests for expected behavior
2. Implement minimal changes
3. Validate with targeted tests first, then broader checks
4. Draft PR with test evidence
"""

write_text(Path("artifacts") / f"issue_{ISSUE_NUMBER}_brief.md", brief)

In [ ]:
# Module 5: execute tests and validation gates
os.chdir(f"/content/{WORKDIR}")

run(f"PYTHONPATH=src:envs .venv/bin/uv run pytest {PYTEST_TARGET} {PYTEST_FLAGS}")

if RUN_LINT:
    run(".venv/bin/uv run usort check src/ tests/")
    run(".venv/bin/uv run ruff format src/ tests/ --check")
    run(".venv/bin/uv run ruff check src/ tests/")

if RUN_FULL_TESTS:
    run("PYTHONPATH=src:envs .venv/bin/uv run pytest tests/ -v --tb=short")

print("Validation module complete")

## Module 6: Git + PR Handoff

> After making code changes in Colab, run the next cell to generate a PR draft artifact.

Then use:
1. GitHub UI or CLI to open/update PR
2. Attach `artifacts/pr_draft_issue_<n>.md` content as your PR body
3. Include test output from Module 5

In [ ]:
# Module 6 (code): generate git summary and PR draft artifact
os.chdir(f"/content/{WORKDIR}")

status = subprocess.check_output("git status --short --branch", shell=True, text=True)
diff_stat = subprocess.check_output("git diff --stat", shell=True, text=True)
recent_commits = subprocess.check_output("git log --oneline -n 5", shell=True, text=True)

pr_body = f"""## Summary
- <replace with concrete change bullets>

## Validation
- PYTHONPATH=src:envs .venv/bin/uv run pytest {PYTEST_TARGET} {PYTEST_FLAGS}

## Evidence
### git status
```
{status.strip()}
```

### git diff --stat
```
{diff_stat.strip() or 'No unstaged diff'}
```

### recent commits
```
{recent_commits.strip()}
```

Refs #{ISSUE_NUMBER}
"""

write_text(Path("artifacts") / f"pr_draft_issue_{ISSUE_NUMBER}.md", pr_body)
print("PR draft artifact generated")